In [ ]:
# ISIC 2019 Ablation — Config 5: Trimodal No Gated Fusion (Concat)

import os, random, warnings, logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import cv2
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

from torchinfo import summary
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

In [ ]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32

train_dir = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/Train_Validation_dataset/content/Train_Validation_dataset/train"
val_dir   = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/Train_Validation_dataset/content/Train_Validation_dataset/val"
test_dir  = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/Test_dataset/content/test_dataset2"

train_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandAugment(num_ops=3, magnitude=9),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.05, 0.25), value=0, inplace=False),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_test_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
from transformers import CLIPTokenizer
from sklearn.preprocessing import StandardScaler

meta_path = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/ISIC_2019_Training_Metadata.csv"
df = pd.read_csv(meta_path)

# Clinical prompt — identical to best ISIC 2019 notebook
def create_clinical_prompt(row):
    age  = str(row.get("age_approx", "unknown"))
    sex  = str(row.get("sex", "patient"))
    site = str(row.get("anatom_site_general", "an unknown body site"))
    return f"A dermoscopic image of a skin lesion located on the {site} of a {age} year old {sex}."

df["clinical_prompt"] = df.apply(create_clinical_prompt, axis=1)

# Tabular: age_approx + sex + anatom_site_general
# Normalise age using training images only (avoid leakage)
# Build train image set from ImageFolder targets
train_images = set(
    os.path.splitext(os.path.basename(p))[0]
    for p, _ in datasets.ImageFolder(train_dir).samples
)
train_mask = df.index.isin(train_images) if df.index.name == "image"              else df["image"].isin(train_images) if "image" in df.columns else pd.Series([True]*len(df))

df_work = df.copy()
if "image" in df_work.columns:
    df_work = df_work.set_index("image")

df_work["age_approx"] = pd.to_numeric(df_work["age_approx"], errors="coerce")
age_mean = df_work.loc[df_work.index.isin(train_images), "age_approx"].mean()
age_std  = df_work.loc[df_work.index.isin(train_images), "age_approx"].std() + 1e-8
df_work["age_norm"] = (df_work["age_approx"].fillna(age_mean) - age_mean) / age_std

for col in ["sex", "anatom_site_general"]:
    df_work[col] = df_work[col].fillna("unknown")

df_work = pd.get_dummies(df_work, columns=["sex", "anatom_site_general"],
                          dummy_na=False, dtype=float)

TABULAR_COLS = ["age_norm"] + [c for c in df_work.columns
                if c.startswith("sex_") or c.startswith("anatom_site_general_")]
NUM_TAB = len(TABULAR_COLS)
print(f"Tabular features: {NUM_TAB} — {TABULAR_COLS}")
print(f"Example prompt: {df_work['clinical_prompt'].iloc[0]}")

In [ ]:
class TriModalImageFolder(datasets.ImageFolder):
    def __init__(self, root, df, tab_cols, transform=None):
        super().__init__(root, transform=transform)
        self.df        = df
        self.tab_cols  = tab_cols
        self.tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
        self._zero_tab = torch.zeros(len(tab_cols))

    def __getitem__(self, index):
        image, target = super().__getitem__(index)
        path, _ = self.samples[index]
        img_name = os.path.splitext(os.path.basename(path))[0]

        if img_name in self.df.index:
            row         = self.df.loc[img_name]
            text_prompt = row["clinical_prompt"]
            tab_feats   = torch.tensor(row[self.tab_cols].values.astype(float),
                                       dtype=torch.float32)
        else:
            text_prompt = "A dermoscopic image of a skin lesion."
            tab_feats   = self._zero_tab

        tokens = self.tokenizer(text_prompt, padding="max_length", max_length=32,
                                truncation=True, return_tensors="pt")
        return (image, tokens["input_ids"].squeeze(0),
                tokens["attention_mask"].squeeze(0), tab_feats, target)

train_ds = TriModalImageFolder(train_dir, df_work, TABULAR_COLS, transform=train_tfms)
val_ds   = TriModalImageFolder(val_dir,   df_work, TABULAR_COLS, transform=val_test_tfms)
test_ds  = TriModalImageFolder(test_dir,  df_work, TABULAR_COLS, transform=val_test_tfms)

train_targets = train_ds.targets
class_counts  = Counter(train_targets)
smoothed      = {cls: np.power(cnt, 0.5) for cls, cnt in class_counts.items()}
sample_w      = torch.DoubleTensor([1.0/smoothed[t] for t in train_targets])
sampler       = WeightedRandomSampler(weights=sample_w,
                                      num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

In [ ]:
class_to_idx = train_ds.class_to_idx
IDX_TO_CLASS = {
    class_to_idx["AK"]:   "Actinic Keratoses (AK)",
    class_to_idx["BCC"]:  "Basal Cell Carcinoma (BCC)",
    class_to_idx["BKL"]:  "Benign Keratosis-like Lesions (BKL)",
    class_to_idx["DF"]:   "Dermatofibroma (DF)",
    class_to_idx["MEL"]:  "Melanoma (MEL)",
    class_to_idx["NV"]:   "Melanocytic Nevi (NV)",
    class_to_idx["SCC"]:  "Squamous Cell Carcinoma (SCC)",
    class_to_idx["VASC"]: "Vascular Lesions (VASC)",
}
NUM_CLASSES = len(IDX_TO_CLASS)
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
print("Train:", len(train_ds), "| Val:", len(val_ds), "| Test:", len(test_ds))
for i, n in enumerate(class_names): print(i, "→", n)

In [ ]:
from transformers import CLIPTextModel

class TriModalNoConcatNet(nn.Module):
    """All three modalities fused by simple concatenation — no learned gate."""
    def __init__(self, num_meta_features, num_classes=8, pretrained=True):
        super().__init__()
        convnext = models.convnext_tiny(
            weights=models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None)
        self.convnext_features = convnext.features

        swin = models.swin_t(
            weights=models.Swin_T_Weights.DEFAULT if pretrained else None)
        self.swin_features = swin.features
        self.swin_norm     = swin.norm

        self.vision_fuse = nn.Sequential(
            nn.Linear(768 + 768, 1024), nn.LayerNorm(1024), nn.GELU(), nn.Dropout(0.3)
        )

        self.text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
        text_dim = self.text_encoder.config.hidden_size

        embed = 256
        self.vis_proj  = nn.Linear(1024, embed)
        self.text_proj = nn.Linear(text_dim, embed)
        self.pos_embed = nn.Parameter(torch.randn(1, 64, embed) * 0.02)

        self.text_to_vis_attn = nn.MultiheadAttention(embed, num_heads=8,
                                                       dropout=0.1, batch_first=True)
        self.vis_to_text_attn = nn.MultiheadAttention(embed, num_heads=8,
                                                       dropout=0.1, batch_first=True)

        self.tabular_mlp = nn.Sequential(
            nn.Linear(num_meta_features, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(inplace=True), nn.Dropout(0.3)
        )
        self.meta_proj = nn.Linear(128, embed)

        # Simple concat of 5 streams → classifier (no gate)
        self.classifier = nn.Sequential(
            nn.Linear(embed * 5, 512),
            nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, input_ids, attention_mask, tab, return_attn=False):
        f_cnxt       = self.convnext_features(x).flatten(2).transpose(1, 2)
        f_swin       = self.swin_norm(self.swin_features(x)).flatten(1, 2)
        fused_vision = self.vision_fuse(torch.cat([f_cnxt, f_swin], dim=-1))

        text_feats = self.text_encoder(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state

        v_emb = F.normalize(self.vis_proj(fused_vision) + self.pos_embed, p=2, dim=-1)
        t_emb = F.normalize(self.text_proj(text_feats), p=2, dim=-1)

        t_att, attn_w = self.text_to_vis_attn(query=t_emb, key=v_emb, value=v_emb)
        v_att, _      = self.vis_to_text_attn(query=v_emb, key=t_emb, value=t_emb)

        streams = [t_att.mean(1), v_att.mean(1),
                   t_emb.mean(1), v_emb.mean(1),
                   self.meta_proj(self.tabular_mlp(tab))]
        fused = torch.cat(streams, dim=-1)   # [B, 1280] — simple concat

        preds = self.classifier(fused)
        if return_attn: return preds, attn_w
        return preds

model = TriModalNoConcatNet(num_meta_features=NUM_TAB, num_classes=NUM_CLASSES).to(DEVICE)
for p in model.convnext_features.parameters(): p.requires_grad = False
for p in model.swin_features.parameters():     p.requires_grad = False
for p in model.swin_norm.parameters():         p.requires_grad = False
for p in model.text_encoder.parameters():      p.requires_grad = False

summary(model, input_size=[(BATCH_SIZE,3,256,256),(BATCH_SIZE,32),(BATCH_SIZE,32),(BATCH_SIZE,NUM_TAB)],
        dtypes=[torch.float32, torch.long, torch.long, torch.float32],
        col_names=["input_size","output_size","num_params"], depth=3)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(reduction="none", label_smoothing=0.1)
    def forward(self, inputs, targets):
        ce = self.ce(inputs, targets)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

In [ ]:
scaler = GradScaler("cuda")

def compute_specificity(y_true, y_pred, n):
    cm = confusion_matrix(y_true, y_pred, labels=range(n))
    s = []
    for i in range(n):
        TP=cm[i,i]; FP=cm[:,i].sum()-TP; FN=cm[i,:].sum()-TP; TN=cm.sum()-(TP+FP+FN)
        s.append(TN/(TN+FP) if (TN+FP)>0 else 0)
    return np.mean(s)

def rand_bbox(size, lam):
    W,H = size[2], size[3]
    cut_w = int(W * np.sqrt(1.-lam)); cut_h = int(H * np.sqrt(1.-lam))
    cx = np.random.randint(W); cy = np.random.randint(H)
    return (np.clip(cx-cut_w//2,0,W), np.clip(cy-cut_h//2,0,H),
            np.clip(cx+cut_w//2,0,W), np.clip(cy+cut_h//2,0,H))

def train_one_epoch(model, loader, optimizer, criterion, device,
                    mixup_prob=1.0, cutmix_prob=0.0):
    model.train()
    running_loss=0.0; yt,yp=[],[]
    for images, input_ids, attn_mask, tab, labels in tqdm(loader, desc="Train", leave=False):
        images=images.to(device); input_ids=input_ids.to(device)
        attn_mask=attn_mask.to(device); tab=tab.to(device); labels=labels.to(device)

        r = np.random.rand()
        optimizer.zero_grad()
        with autocast("cuda"):
            if r < cutmix_prob:
                lam = np.random.beta(1.0, 1.0)
                idx = torch.randperm(images.size(0), device=device)
                ya, yb = labels, labels[idx]
                x1,y1,x2,y2 = rand_bbox(images.size(), lam)
                mix = images.clone(); mix[:,:,x1:x2,y1:y2] = images[idx,:,x1:x2,y1:y2]
                lam = 1 - ((x2-x1)*(y2-y1)/(images.size(-1)*images.size(-2)))
                out = model(mix, input_ids, attn_mask, tab)
                loss = lam*criterion(out,ya) + (1-lam)*criterion(out,yb)
            elif r < cutmix_prob + mixup_prob:
                lam = np.random.beta(0.3, 0.3)
                idx = torch.randperm(images.size(0), device=device)
                ya, yb = labels, labels[idx]
                mix = lam*images + (1-lam)*images[idx]
                mix_tab = lam*tab + (1-lam)*tab[idx]
                out = model(mix, input_ids, attn_mask, mix_tab)
                loss = lam*criterion(out,ya) + (1-lam)*criterion(out,yb)
            else:
                out = model(images, input_ids, attn_mask, tab)
                ya  = labels
                loss = criterion(out, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        running_loss += loss.item()*images.size(0)
        yt.extend(ya.cpu().numpy()); yp.extend(out.argmax(1).cpu().numpy())
    L=running_loss/len(yt)
    return (L, accuracy_score(yt,yp),
            precision_score(yt,yp,average="macro",zero_division=0),
            recall_score(yt,yp,average="macro",zero_division=0),
            f1_score(yt,yp,average="macro",zero_division=0),
            compute_specificity(yt,yp,NUM_CLASSES))

def evaluate(model, loader, criterion, device):
    model.eval(); running_loss=0.0; yt,yp=[],[]
    with torch.no_grad():
        for images, input_ids, attn_mask, tab, labels in tqdm(loader, desc="Eval", leave=False):
            images=images.to(device); input_ids=input_ids.to(device)
            attn_mask=attn_mask.to(device); tab=tab.to(device); labels=labels.to(device)
            out = model(images, input_ids, attn_mask, tab)
            if isinstance(out, tuple): out = out[0]
            loss = criterion(out, labels)
            running_loss += loss.item()*images.size(0)
            yt.extend(labels.cpu().numpy()); yp.extend(out.argmax(1).cpu().numpy())
    L=running_loss/len(yt)
    return (L, accuracy_score(yt,yp),
            precision_score(yt,yp,average="macro",zero_division=0),
            recall_score(yt,yp,average="macro",zero_division=0),
            f1_score(yt,yp,average="macro",zero_division=0),
            compute_specificity(yt,yp,NUM_CLASSES), yt, yp)

In [ ]:
criterion = FocalLoss(gamma=2.0)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-2
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

PHASE1_EPOCHS = 10
train_losses, val_losses = [], []
best_val_f1=0.0; best_model_state=None

for epoch in range(PHASE1_EPOCHS):
    print(f"\nPhase 1 Epoch [{epoch+1}/{PHASE1_EPOCHS}]")
    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl = evaluate(model, val_loader, criterion, DEVICE)
    train_losses.append(tr[0]); val_losses.append(vl[0])
    scheduler.step()
    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} F1:{tr[4]:.4f} Spec:{tr[5]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} F1:{vl[4]:.4f} Spec:{vl[5]:.4f}")
    if vl[4] > best_val_f1:
        best_val_f1=vl[4]; best_model_state=model.state_dict()
        print(f"Val F1 improved ({best_val_f1:.4f}) — saving")

if best_model_state: model.load_state_dict(best_model_state)
print(f"Phase 1 done. Best val F1: {best_val_f1:.4f}")
for p in model.parameters(): p.requires_grad = True

In [ ]:
optimizer = torch.optim.AdamW([
    {"params": model.convnext_features.parameters(), "lr": 1e-6},
    {"params": model.swin_features.parameters(),     "lr": 1e-6},
    {"params": model.swin_norm.parameters(),         "lr": 1e-6},
    {"params": model.text_encoder.parameters(),      "lr": 1e-6},
    {"params": model.vision_fuse.parameters(),       "lr": 1e-4},
    {"params": model.vis_proj.parameters(),          "lr": 1e-4},
    {"params": model.text_proj.parameters(),         "lr": 1e-4},
    {"params": [model.pos_embed],                    "lr": 1e-4},
    {"params": model.text_to_vis_attn.parameters(),  "lr": 1e-4},
    {"params": model.vis_to_text_attn.parameters(),  "lr": 1e-4},
    {"params": model.tabular_mlp.parameters(),       "lr": 1e-4},
    {"params": model.meta_proj.parameters(),         "lr": 1e-4},
    {"params": model.classifier.parameters(),        "lr": 1e-4},
], weight_decay=5e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)

PHASE2_EPOCHS = 60
fine_train_losses, fine_val_losses = [], []
RESTART_EPOCHS = {10, 30}
patience=15; counter=0

for epoch in range(PHASE2_EPOCHS):
    print(f"\nPhase 2 Epoch [{epoch+1}/{PHASE2_EPOCHS}]")
    if (epoch+1) in RESTART_EPOCHS: counter=0; print("LR restart — patience reset")
    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl = evaluate(model, val_loader, criterion, DEVICE)
    fine_train_losses.append(tr[0]); fine_val_losses.append(vl[0])
    scheduler.step()
    print(f"LR (backbone):{optimizer.param_groups[0]['lr']:.2e} | LR (head):{optimizer.param_groups[-1]['lr']:.2e}")
    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} F1:{tr[4]:.4f} Spec:{tr[5]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} F1:{vl[4]:.4f} Spec:{vl[5]:.4f}")
    if vl[4] > best_val_f1:
        best_val_f1=vl[4]; best_model_state=model.state_dict()
        torch.save(best_model_state, "/kaggle/working/ISIC2019_C5_TriModalNoGate_Best.pth")
        counter=0; print(f"Val F1 improved ({best_val_f1:.4f}) — saving")
    else:
        counter+=1
        if counter>=patience: print(f"Early stopping at epoch {epoch+1}"); break

if best_model_state: model.load_state_dict(best_model_state)
print(f"Phase 2 done. Best val F1: {best_val_f1:.4f}")

In [ ]:
def print_report(split, model, loader, criterion):
    L,acc,prec,rec,f1,spec,yt,yp = evaluate(model, loader, criterion, DEVICE)
    print(f"\n{'='*50}\n{split.upper()} REPORT")
    print(f"Loss:{L:.4f} | Acc:{acc:.4f} | Spec:{spec:.4f}")
    print(classification_report(yt, yp, target_names=class_names, zero_division=0, digits=4))
    mprec=precision_score(yt,yp,average="macro",zero_division=0)
    mrec=recall_score(yt,yp,average="macro",zero_division=0)
    mf1=f1_score(yt,yp,average="macro",zero_division=0)
    print(f"Macro → Prec:{mprec:.4f} | Rec:{mrec:.4f} | F1:{mf1:.4f} | Spec:{spec:.4f}")
    return yt, yp

train_yt,train_yp = print_report("Train",      model, train_loader, criterion)
val_yt,val_yp     = print_report("Validation", model, val_loader,   criterion)
test_yt,test_yp   = print_report("Test",       model, test_loader,  criterion)

In [ ]:
from scipy import stats as scipy_stats

# ── Change this per notebook ─────────────────────────────────
DATASET_NAME = "ISIC 2019"   # "HAM10000" | "PAD-UFES-20" | "ISIC 2019"
FILE_TAG     = "isic"        # "ham"       | "pad"          | "isic"

N_BOOTSTRAP = 1000
rng         = np.random.default_rng(42)
y_true_arr  = np.array(test_yt)
y_pred_arr  = np.array(test_yp)
n_test      = len(y_true_arr)

boot = {"Accuracy": [], "F1 (Macro)": [], "Precision (Macro)": [], "Recall (Macro)": [], "Specificity": []}
per_class_f1 = {name: [] for name in class_names}

def compute_specificity_boot(y_t, y_p, num_classes):
    cm = confusion_matrix(y_t, y_p, labels=list(range(num_classes)))
    specs = []
    for i in range(num_classes):
        TP = cm[i, i]; FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP; TN = cm.sum() - (TP + FP + FN)
        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return float(np.mean(specs))

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_test, size=n_test)
    y_t, y_p = y_true_arr[idx], y_pred_arr[idx]
    boot["Accuracy"].append(accuracy_score(y_t, y_p))
    boot["F1 (Macro)"].append(f1_score(y_t, y_p, average="macro", zero_division=0))
    boot["Precision (Macro)"].append(precision_score(y_t, y_p, average="macro", zero_division=0))
    boot["Recall (Macro)"].append(recall_score(y_t, y_p, average="macro", zero_division=0))
    boot["Specificity"].append(compute_specificity_boot(y_t, y_p, NUM_CLASSES))
    pc = f1_score(y_t, y_p, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    for ci, name in enumerate(class_names):
        per_class_f1[name].append(pc[ci])

def ci95(arr): return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

print("\n" + "=" * 72)
print(f"  {DATASET_NAME} — C5 Trimodal No Gate — Bootstrap CI (n={N_BOOTSTRAP})")
print("=" * 72)
print(f"{'Metric':<25} {'Mean':>8}  {'Std':>8}  {'95% CI Lower':>14}  {'95% CI Upper':>14}")
print("=" * 72)
for metric, values in boot.items():
    arr = np.array(values); lo, hi = ci95(arr)
    print(f"{metric:<25} {arr.mean():>8.4f}  {arr.std():>8.4f}  {lo:>14.4f}  {hi:>14.4f}")
print("-" * 72)
print(f"\n{'Class':<52} {'Mean':>8}  {'Std':>8}  {'95% CI Lower':>14}  {'95% CI Upper':>14}")
print("-" * 72)
for name in class_names:
    arr = np.array(per_class_f1[name]); lo, hi = ci95(arr)
    print(f"{name:<52} {arr.mean():>8.4f}  {arr.std():>8.4f}  {lo:>14.4f}  {hi:>14.4f}")
print("=" * 72)

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
fig.subplots_adjust(top=0.82, wspace=0.4)
for ax, (label, values) in zip(axes, boot.items()):
    arr = np.array(values); lo, hi = ci95(arr); mn, sd = arr.mean(), arr.std()
    ax.boxplot(arr, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor="#4c72b0", alpha=0.75),
               medianprops=dict(color="white", linewidth=2),
               whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
               flierprops=dict(marker="o", markersize=3, markerfacecolor="gray", alpha=0.5))
    ax.set_title(label, fontsize=10, fontweight="bold", pad=8)
    ax.set_xticks([]); ax.set_ylabel("Score", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.text(1, ax.get_ylim()[1], f"μ={mn:.4f}\nσ={sd:.4f}\n95%CI [{lo:.4f},{hi:.4f}]",
            ha="center", va="top", fontsize=7.5, color="#222222",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray", alpha=0.8))
plt.suptitle(f"{DATASET_NAME} C5 — Bootstrap Metric Distributions (n={N_BOOTSTRAP}) — Test Set",
             fontsize=13, fontweight="bold", y=0.97)
plt.savefig(f"/kaggle/working/{FILE_TAG}_c5_bootstrap_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

means  = np.array([np.mean(per_class_f1[n]) for n in class_names])
stds   = np.array([np.std(per_class_f1[n])  for n in class_names])
lo_arr = np.array([ci95(per_class_f1[n])[0] for n in class_names])
hi_arr = np.array([ci95(per_class_f1[n])[1] for n in class_names])

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, means, yerr=[means - lo_arr, hi_arr - means], capsize=6,
              color="#4c72b0", alpha=0.8, ecolor="#333333",
              error_kw={"linewidth": 1.8, "capthick": 1.8}, width=0.55)
ax.set_xticks(x)
ax.set_xticklabels([n.split("(")[-1].replace(")", "").strip() if "(" in n else n
                    for n in class_names], fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11); ax.set_ylim(0, 1.15)
ax.set_title(f"{DATASET_NAME} C5 — Per-Class F1 Mean ± 95% CI (Bootstrap)",
             fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3, linestyle="--"); ax.spines[["top", "right"]].set_visible(False)
for bar, m, s, hi in zip(bars, means, stds, hi_arr):
    ax.text(bar.get_x() + bar.get_width() / 2, hi + 0.025,
            f"{m:.3f}\n±{s:.3f}", ha="center", va="bottom", fontsize=9, color="#111111")
plt.tight_layout()
plt.savefig(f"/kaggle/working/{FILE_TAG}_c5_bootstrap_perclass_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cm = confusion_matrix(test_yt, test_yp)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("ISIC 2019 Test Confusion Matrix — Config 5: Trimodal No Gated Fusion")
plt.tight_layout(); plt.show()

In [ ]:
def gradcam_on_convnext(model, loader, class_names, num_images=6):
    model.eval()
    activations, gradients = {}, {}
    def fwd_hook(m, inp, out): activations["feat"] = out.detach()
    def bwd_hook(m, g_in, g_out): gradients["feat"] = g_out[0].detach()
    handle_f = model.convnext_features[-1].register_forward_hook(fwd_hook)
    handle_b = model.convnext_features[-1].register_full_backward_hook(bwd_hook)

    batch   = next(iter(loader))
    images  = batch[0][:num_images].to(DEVICE)
    ids     = batch[1][:num_images].to(DEVICE)
    masks   = batch[2][:num_images].to(DEVICE)
    tab     = batch[3][:num_images].to(DEVICE)
    labels  = batch[4][:num_images]

    model.zero_grad()
    with torch.enable_grad():
        out      = model(images, ids, masks, tab)
        if isinstance(out, tuple): out = out[0]
        pred_cls = out.argmax(dim=1)
        out[range(len(images)), pred_cls].sum().backward()

    handle_f.remove(); handle_b.remove()
    acts    = activations["feat"]
    grads   = gradients["feat"]
    weights = grads.mean(dim=(2,3), keepdim=True)
    cam     = F.relu((weights * acts).sum(dim=1))

    mean = np.array([0.485,0.456,0.406])
    std  = np.array([0.229,0.224,0.225])
    fig, axes = plt.subplots(num_images, 2, figsize=(10, 4*num_images))
    for i in range(num_images):
        img = np.clip(std * images[i].cpu().permute(1,2,0).numpy() + mean, 0, 1)
        c   = cam[i].cpu().numpy()
        c   = (c - c.min()) / (c.max() + 1e-8)
        c   = cv2.resize(c, (img.shape[1], img.shape[0]))
        axes[i,0].imshow(img); axes[i,0].set_title(f"True: {class_names[labels[i]]}", fontsize=9); axes[i,0].axis("off")
        axes[i,1].imshow(img); axes[i,1].imshow(c, cmap="jet", alpha=0.5)
        axes[i,1].set_title(f"Pred: {class_names[pred_cls[i].item()]}", fontsize=9); axes[i,1].axis("off")
    plt.suptitle("GradCAM — ConvNeXt-Tiny (Config 5: Trimodal No Gated Fusion)", fontsize=12)
    plt.tight_layout(); plt.show()

gradcam_on_convnext(model, test_loader, class_names, num_images=6)